# Phase 10.2 — Ranking Features

Extract raw `RankingFeatures` from a ranking candidate and injected product context.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.adapters import ranking_request_from_retrieval_response
from productiq.ranking.feature_extractor import extract_features_for_request
from productiq.ranking.product_context import RankingProductContext
from productiq.representation.builder import build_product_representation
from productiq.representation.filtering import build_filtering_representation_from_canonical
from productiq.representation.query_contract import build_query_representation
from productiq.retrieval.contracts import RetrievalCandidate, RetrievalMethod, RetrievalResponse
from tests.database.catalog_fixtures import make_product_record

In [ ]:
record = make_product_record(product_id="demo-001", brand_normalized="nike")
product = build_product_representation(record)
filtering = build_filtering_representation_from_canonical(record)
context = RankingProductContext(product_id=product.product_id, product=product, filtering=filtering)
query = build_query_representation("running shoes")
retrieval = RetrievalResponse(
    candidates=(RetrievalCandidate(product_id=product.product_id, score=0.9, method=RetrievalMethod.VECTOR),),
)
request = ranking_request_from_retrieval_response(query=query, response=retrieval, top_k=5)
features = extract_features_for_request(request, product_context_by_id={product.product_id: context})
features[0].model_dump(mode="json")